# 02 — Fingerprint generation
Run in order. This notebook prepares one explicit saved split without retrieval.

In [1]:
from pathlib import Path
import sys, json, numpy as np
from rdkit import rdBase
from chemtellect.fingerprints import load_configuration
from chemtellect.fingerprint_store import validate_split, prepare_artifact, load_artifact, unpack_fingerprints, pack_fingerprints
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p/'src/chemtellect').is_dir()), None)
if ROOT is None: raise RuntimeError('Start JupyterLab within the repository.')
print(ROOT, sys.executable, sys.version, np.__version__, rdBase.rdkitVersion)

/home/ruben/Coding Projects/chemtellect /home/ruben/Coding Projects/chemtellect/.venv/bin/python3 3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0] 2.5.4 2026.09.1


## Select one saved split
Change the explicit path for another approved saved pool. Never automatically select alternatives.

In [2]:
split_path = ROOT/'data/processed/chemicalspace_new/package_00/split_f85552ea5b17e482.json'
config_path = ROOT/'experiments/configs/morgan_r2_2048_chiral_v1.json'
if not split_path.is_file(): raise FileNotFoundError(f'Required saved split absent: {split_path}; complete and review the overview split stage first.')
config = load_configuration(config_path)
BATCH_SIZE = 1000
print(json.dumps(config, indent=2))
print({'batch_size': BATCH_SIZE, 'thread_count': 1})

{
  "schema_version": 1,
  "generator": {
    "radius": 2,
    "fpSize": 2048,
    "includeChirality": true,
    "useBondTypes": true,
    "includeRingMembership": true,
    "countSimulation": false,
    "countBounds": null,
    "onlyNonzeroInvariants": false,
    "includeRedundantEnvironments": false,
    "atomInvariantsGenerator": null,
    "bondInvariantsGenerator": null
  },
  "preprocessing": {
    "input": "canonical_smiles",
    "sanitize": true,
    "hydrogens": "SMILES parser handling; no AddHs or RemoveHs stage",
    "standardization": "none",
    "disconnected_components": "keep",
    "stereochemistry": "preserve specified; leave unspecified",
    "atom_invariants": "standard Morgan connectivity",
    "atom_centers": "all",
    "ignored_atoms": [],
    "custom_invariants": null,
    "output": "GetFingerprint ExplicitBitVect",
    "source_rfp": "never used",
    "failure": "stop with role, source_id and line_number",
    "fingerprint_deduplication": false
  },
  "rationale": 

In [3]:
split = validate_split(json.loads(split_path.read_text()))
print({k:v for k,v in split.items() if k not in ('queries','index_order')})

{'canonicalization': {'canonical': True, 'isomericSmiles': True, 'sanitize': True, 'standardization': 'none'}, 'eligible_unique_count': 1200, 'index_sizes': [1000], 'ordering_algorithm': 'sha256-json-seed-source-package-id-v1', 'package': 'package_00', 'query_count': 100, 'raw_sha256': 'f3b298720497eb4ada39fa0dcfdf85f2c5851922ea8d0423c6d68d45f45c0f39', 'rdkit_version': '2026.09.1', 'representative_policy': 'first raw physical line per canonical structure', 'seed': 42, 'source': 'chemicalspace_new', 'split_id': 'f85552ea5b17e4829a09df82c79ad3bb2d762ee672c0f52de234dc7fc668e908', 'split_schema_version': 1, 'unused_count': 100}


## Prepare or verify reuse
Progress is displayed after each generated batch. Reuse performs integrity checks without molecular generation.

In [4]:
result = prepare_artifact(ROOT, split_path, config, BATCH_SIZE,
                          progress=lambda role, done, total: print(f'{role}: {done}/{total}'))
print(json.dumps(result['report'], indent=2))
print('Dimensions:', result['manifest']['dimensions'])
print('Stored bytes:', sum(f['bytes'] for f in result['manifest']['files'].values()))

{
  "artifact_id": "628c36671feeb38e1a97a34dd664bb54f43851374fcea4190a474479a12df427",
  "operation": "reused",
  "start_utc": "2026-11-10T02:02:41.142145+00:00",
  "end_utc": "2026-11-10T02:02:41.159973+00:00",
  "environment": {
    "python": "3.12.3",
    "rdkit": "2026.09.1",
    "numpy": "2.5.4",
    "git_revision": "2131d4b14065c5924312332d0b092a736cf3d37f",
    "git_dirty": true
  },
  "timings_ns": {
    "verification": 7448062,
    "generator_initialization": null,
    "artifact_writing": null,
    "reload_verification": 10189505,
    "queries_parsing": null,
    "queries_generation": null,
    "queries_packing": null,
    "index_parsing": null,
    "index_generation": null,
    "index_packing": null,
    "total": 17816611
  },
  "execution": {
    "batch_size": 1000,
    "thread_count": 1
  },
  "completed_counts": {
    "queries": 100,
    "index": 1000
  }
}
Dimensions: {'index': [1000, 256], 'queries': [100, 256]}
Stored bytes: 374224


## Exact reconstruction and identity alignment
Inspect one row without computing similarities. The split is authoritative for SMILES.

In [5]:
queries, index = load_artifact(result['artifact_path'], split_path, config, ROOT, split['index_sizes'][0])
reconstructed = unpack_fingerprints(queries[:1])
assert np.array_equal(pack_fingerprints(reconstructed), queries[:1])
records = json.loads((result['artifact_path']/'records.json').read_text())
assert records['queries'][0]['source_id'] == split['queries'][0]['source_id']
print(queries.shape, index.shape, records['queries'][0], reconstructed[0].GetNumBits())
print('Artifact:', result['artifact_path'])
print('Run report:', result['report_path'])

(100, 256) (1000, 256) {'line_number': 173, 'row_index': 0, 'source_id': 'objectid:5bee9fbef127007b1bbcda24'} 2048
Artifact: /home/ruben/Coding Projects/chemtellect/data/processed/chemicalspace_new/package_00/fingerprints/628c36671feeb38e1a97a34dd664bb54f43851374fcea4190a474479a12df427
Run report: /home/ruben/Coding Projects/chemtellect/experiments/results/fingerprints/chemicalspace_new/package_00/628c36671feeb38e1a97a34dd664bb54f43851374fcea4190a474479a12df427/run_06174604da314ed6bc93702b817b7462.json


## Next checkpoint
Exact retrieval will consume these reconstructed bits. NMSLIB will later consume a lossless encoding of the same bits, without parsing SMILES. Serialization and distance tolerance remain for review. No retrieval, ground truth or recall is executed here. See `docs/fingerprints.md` for timing boundaries, scientific limitations and interruption recovery.